[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/module_06_confidentiality_attacks/Lab_6b_Model_Inversion_Attacks_and_Defenses.ipynb)


# **Lab 6b: Model Inversion Attacks and Defenses**

**Course:** Introduction to Data Security Pr.  
**Module 6:**  Confidentiality Attacks  
**Estimated Time:** 45 minutes

---

## Learning Objectives

By the end of this lab, you will understand:

1. **Model Inversion:** Reconstructing training data from model parameters/outputs
2. **Privacy Extraction:** How gradients leak information about training samples
3. **Attack Vectors:** Feature inversion and sample reconstruction
4. **Threat Model:** White-box vs black-box inversion attacks
5. **Real-World Impact:** Reconstructing faces, names, and sensitive attributes from ML models

## Table of Contents

1. [Threat Model: Data Reconstruction](#threat-model)
2. [Feature Inversion: Basic Attack](#feature-inversion)
3. [Gradient-Based Sample Reconstruction](#gradient-reconstruction)
4. [Exercises](#exercises)

## Threat Model: Data Reconstruction <a id="threat-model"></a>

- Can an attacker reconstruct training data from model outputs/parameters?

### Real-World Scenarios:

| Scenario | Attack | Data at Risk | Impact |
|----------|--------|--------------|--------|
| **Face Recognition** | Reconstruct faces | Biometric data | Identity theft |
| **Medical Records** | Reconstruct medical history | Patient data | Privacy violation |
| **Credit Scoring** | Infer financial attributes | Income/credit data | Discriminatory use |

### Why it leaks data:

1. **Model Capacity:** Bigger models usually remember more detail.
2. **Training Data Uniqueness:** Rare or unusual samples are easier to recover.
3. **Confidence Signals:** High-confidence outputs often reveal what the model has memorized.

## Model Inversion Theory <a id="theory"></a>

A trained model can leak visual clues about its training data when an attacker asks the right questions.

### Why it matters:

These attacks may not recover the exact original sample, but they can still reveal recognizable structure and sensitive attributes.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List
import torch.nn.functional as F
from sklearn.datasets import load_breast_cancer, fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Gradient Ascent Reconstruction on Faces

Try to compare the average face for each subject with a reconstruction obtained by gradient ascent.

In [ ]:
faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X_faces = faces.data
y_faces = faces.target

N_SUBJECTS = 10
subject_mask = y_faces < N_SUBJECTS
X_faces = X_faces[subject_mask]
y_faces = y_faces[subject_mask]

Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(
    X_faces, y_faces, test_size=0.2, random_state=42, stratify=y_faces
)

In [ ]:
class FaceNet(nn.Module):
    def __init__(self, in_dim=4096, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
Xt = torch.tensor(Xf_tr, dtype=torch.float32, device=device)
yt = torch.tensor(yf_tr, dtype=torch.long, device=device)
Xv = torch.tensor(Xf_te, dtype=torch.float32, device=device)
yv = torch.tensor(yf_te, dtype=torch.long, device=device)

face_model = FaceNet(n_classes=N_SUBJECTS).to(device)
face_optimizer = optim.Adam(face_model.parameters(), lr=1e-3, weight_decay=1e-4)
face_criterion = nn.CrossEntropyLoss()

EPOCHS = 50
for epoch in range(EPOCHS):
    face_model.train()
    face_optimizer.zero_grad()
    face_loss = face_criterion(face_model(Xt), yt)
    face_loss.backward()
    face_optimizer.step()
    if epoch % 10 == 0:
        print(f'Epoch {epoch:2d}/{EPOCHS} - loss: {face_loss.item():.4f}')

face_model.eval()
with torch.no_grad():
    tr_acc = (face_model(Xt).argmax(1) == yt).float().mean().item()
    te_acc = (face_model(Xv).argmax(1) == yv).float().mean().item()
print(f'Face model — Train acc: {tr_acc:.3f}, Test acc: {te_acc:.3f}')

### Model Inversion Attack using Gradient Ascent

The goal is to "invert" the model's knowledge to reconstruct what it thinks a specific class (in this case, a person's face) looks like, starting from pure random noise.

In [ ]:
def invert_image(model, target_class, n_iter=2000, lr=0.01, tv_weight=1e-4, l2_weight=1e-3):
    model.eval()

    # tensor of random noise
    x = torch.rand(1, 4096, requires_grad=True, device=device)
    opt = optim.Adam([x], lr=lr)
    
    for i in range(n_iter):
        opt.zero_grad()
        logits = model(x)
        loss_cls = -logits[0, target_class]
        loss_l2 = l2_weight * (x ** 2).sum()
        img = x.view(1, 64, 64)
        tv = ((img[:, 1:, :] - img[:, :-1, :]) ** 2).sum() + ((img[:, :, 1:] - img[:, :, :-1]) ** 2).sum()
        loss_tv = tv_weight * tv
        loss = loss_cls + loss_l2 + loss_tv
        loss.backward()
        opt.step()
        with torch.no_grad():
            x.clamp_(0, 1)
        if (i + 1) % 500 == 0:
            print(f'  Iteration {i + 1}/{n_iter} — confidence={torch.softmax(logits, dim=1)[0, target_class].item():.4f}')
    return x.detach().cpu().numpy().reshape(64, 64)

In [ ]:
TARGET_SUBJECTS = [0, 1, 2, 3]
inverted_faces = {}

print(f'Starting Model Inversion (Gradient Ascent) for {len(TARGET_SUBJECTS)} subjects...')

for subject_id in TARGET_SUBJECTS:
    print(f'\n>>> Inverting Subject ID: {subject_id}')
    inverted_faces[subject_id] = invert_image(face_model, subject_id)

In [ ]:
fig, axes = plt.subplots(2, len(TARGET_SUBJECTS), figsize=(3 * len(TARGET_SUBJECTS), 6))

for col, subj in enumerate(TARGET_SUBJECTS):
    true_mean_face = Xf_tr[yf_tr == subj].mean(axis=0).reshape(64, 64)
    axes[0, col].imshow(true_mean_face, cmap='gray')
    axes[0, col].set_title(f'Subject {subj}\nTrue mean')
    axes[0, col].axis('off')

    axes[1, col].imshow(inverted_faces[subj], cmap='gray')
    axes[1, col].set_title(f'Subject {subj}\nInverted')
    axes[1, col].axis('off')

plt.suptitle('Model Inversion — True Class Mean vs Gradient-Ascent Reconstruction', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Exercises <a id="exercises"></a>

### Exercise 1: Feature Inversion Optimization
Improve feature inversion for image data by:
- Varying TV weight (0.01, 0.1, 1.0)
- Adding frequency regularization (prefer low frequencies)
- Using different initialization (zero, Gaussian, real image perturbation)

Which combination produces most realistic images?

### Exercise 2: Model Inversion on Tabular Data
Perform a model inversion attack on the Breast Cancer dataset:
1. Load the dataset using `sklearn.datasets.load_breast_cancer` and normalize the features.
2. Train a simple MLP classifier (e.g., 2 hidden layers) to achieve >90% accuracy.
3. Implement the `invert_feature_vector` function using Gradient Ascent (as we did for images) to reconstruct a feature vector that maximizes the model's confidence for a target class.
4. Compare the reconstructed feature vector with the actual mean feature vector of the target class in the training set. Which features are reconstructed most accurately?

### Exercise 3: Defense Cost-Benefit Analysis
For production ML system:
- Model: Face recognition (high utility importance)
- Threat: Model inversion attacks
- Budget: 5% accuracy loss maximum

Design defense strategy that maximizes privacy within accuracy constraint.
Compare: Gradient clipping vs DP-SGD vs ensemble approaches